# Tram Lines
**ROUTE OVERVIEW & STOP SEQUENCES 2025**

---

## Table of Contents

- [Architecture](#architecture)
- [Data Loading](#data-loading)
- [Overview](#overview)
- [Line Directory](#line-directory)
- [Helper Functions](#helper-functions)
- [Export](#export)
- [Line Details](#line-details)

## Architecture

### Starting Situation

VBZ Zürich veröffentlicht offene GTFS-Daten (General Transit Feed Specification) als relationale Tabellen, hier als Parquet gespeichert. Um zu wissen „welche Haltestellen bedient Linie X in welcher Reihenfolge?", müssen vier separate Dateien in Reihe verknüpft werden: **routes → trips → stop_times → stops**.

Drei Probleme erschweren das:

**1 · stop_id-Formatinkompatibilität**
Die zusammengeführte `gtfs_tram_stop_times.parquet` (alle Jahre) verwendet das stop_id-Format `gen:23026:…`, während `gtfs_tram_stops.parquet` das Format `ch:1:sloid:…` verwendet — kein Join möglich. Lösung: `gtfs_stop_times_2025.parquet` (jahresspezifische Datei) verwenden, die das korrekte Format hat.

**2 · Kurzläufer**
Eine Linie fährt pro Tag Dutzende Trips. Nicht alle decken die vollständige Strecke ab — manche wenden vorzeitig an Zwischenhalten. Lösung: für jede (Linie, Richtung) den Trip mit den meisten Halten wählen → immer die Hauptstrecke, nie ein Kurzläufer.

**3 · Typinkonsistenzen zwischen Dateien**
- `year`: String `'2025'` in routes/trips/stops/shapes, Int16 `2025` in stop_times_2025
- `direction_id`: Int64 (Werte 0 oder 1) — Polars-Filter brauchen Integer-Vergleich (`== 0`, nicht `== '0'`)

---

### Data Sources

| Datenbedarf | Datei | Pfad | year-Typ |
|:---|:---|:---|:---|
| Tramlinien (route_id, Liniennummer) | `gtfs_tram_routes.parquet` | `data/raw/gtfs/` | String `'2025'` |
| Trips: shape_id, direction_id, Endstation | `gtfs_tram_trips.parquet` | `data/raw/gtfs/` | String `'2025'` |
| Haltestellenreihenfolge | `gtfs_stop_times_2025.parquet` | `data/raw/gtfs/` | Int16 `2025` |
| Haltestellen-Koordinaten | `gtfs_tram_stops.parquet` | `data/raw/gtfs/` | String `'2025'` |
| Streckengeometrie (~300 Punkte/Linie) | `gtfs_tram_shapes.parquet` | `data/raw/gtfs/` | String `'2025'` |
| Offizielle Linienfarben | `LINE_COLORS` | `src/zh_tram_flow/config.py` | — |

> **Nicht verwenden:** `gtfs_tram_stop_times.parquet` (zusammengeführt, alle Jahre) — stop_id-Format `gen:23026:…` ist inkompatibel mit `gtfs_tram_stops.parquet` (`ch:1:sloid:…`). Kein Join möglich.

---

### Derivation Chain

```
routes      year='2025'  →  17 Tramlinien (route_type=0, bereits vorgefiltert)
              ↓ route_id
trips       route_id  →  viele Trips pro Linie + Richtung
              direction_id: 0 / 1  (Int64 — willkürliche Labels, kein festes Hin/Rück)
              trip_headsign:  Zielanzeige vorne am Tram — einziges zuverlässiges Richtungs-Label
              shape_id:  Verweis auf geglättete Streckengeometrie
              ↓ repräsentativer Trip = Trip mit meisten Halten (Hauptstrecke, kein Kurzläufer)
stop_times  trip_id  →  stop_id sortiert nach stop_sequence  (year = 2025 als Int16)
              ↓ stop_id  (Format: ch:1:sloid:… — matcht gtfs_tram_stops ✓)
stops       stop_id  →  stop_name, stop_lat, stop_lon
shapes      shape_id  →  geglättete Koordinatenpunkte für die Kartenlinie
```

---

### Note on Asymmetric Stop Counts

Nicht alle Linien bedienen in beiden Fahrtrichtungen dieselben Haltestellen. Häufige Ursachen:

| Muster | Δ Halte | Erklärung |
|:---|:---:|:---|
| Einbahnstraßen-Routing | 1–3 | Trams nehmen in der Innenstadt je nach Richtung unterschiedliche Wege — andere Straße, andere Haltestellen |
| Schleife an der Endstation | 1–2 | Das Tram dreht an einer Endstation über eine Schleife und passiert dabei einen extra Halt |
| Strukturelle Asymmetrie (Linie 8) | 10 | Der Wollishofen-Abschnitt ist im GTFS 2025 nur für direction_id=1 hinterlegt — kein direction_id=0-Trip deckt die volle Strecke ab. Die gezeigte Strecke (Klusplatz ↔ Wollishoferplatz) war die offizielle L8-Route bis 13. Dezember 2025. Am 14.12.2025 trat der grösste Fahrplanwechsel in der VBZ-Geschichte in Kraft: L8 fährt seitdem Hardturm ↔ Kirche Fluntern. Unsere GTFS-Daten 2025 bilden korrekt den Zeitraum vor diesem Wechsel ab — methodisch konsistent mit dem Verspätungs-Hauptdatensatz (2023–2025). |

## Data Loading

In [ ]:
import pandas as pd
import polars as pl
from IPython.display import display, HTML
import plotly.graph_objects as go

from zh_tram_flow.config import PATHS, LINE_COLORS, LINE_TEXT_COLORS

RAW_GTFS = PATHS['raw'] / 'gtfs'

# ── Tram lines (17 lines, route_type=0 already pre-filtered, year='2025') ──
routes_df = (
    pl.scan_parquet(RAW_GTFS / 'gtfs_tram_routes.parquet')
    .filter(pl.col('year') == '2025')
    .select(['route_id', 'route_short_name'])
    .collect()
)

# ── Trips: direction_id is Int64 (0 or 1) ───────────────────────────────────
trips_df = (
    pl.scan_parquet(RAW_GTFS / 'gtfs_tram_trips.parquet')
    .filter(pl.col('year') == '2025')
    .join(routes_df.lazy(), on='route_id')
    .select(['trip_id', 'route_id', 'route_short_name', 'direction_id', 'trip_headsign', 'shape_id'])
    .collect()
)

# ── Stop coordinates (year='2025', deduplicated) ─────────────────────────────
stops_df = (
    pl.scan_parquet(RAW_GTFS / 'gtfs_tram_stops.parquet')
    .filter(pl.col('year') == '2025')
    .select(['stop_id', 'stop_name', 'stop_lat', 'stop_lon'])
    .collect()
    .unique(subset=['stop_id'])
)
stop_lookup = {r['stop_id']: r for r in stops_df.to_dicts()}

# ── Stop order — filter tram trips from 6M-row parquet ──────────────────────
tram_trip_ids = trips_df['trip_id'].to_list()
stop_times_df = (
    pl.scan_parquet(RAW_GTFS / 'gtfs_stop_times_2025.parquet')
    .filter(pl.col('trip_id').is_in(tram_trip_ids))
    .select(['trip_id', 'stop_id', 'stop_sequence'])
    .sort(['trip_id', 'stop_sequence'])
    .collect()
)

# ── Representative trip per (line, direction): most stops = main route ───────
stop_counts = (
    stop_times_df
    .group_by('trip_id')
    .agg(pl.len().alias('n_stops'))
)
rep_trips = (
    trips_df
    .join(stop_counts, on='trip_id', how='left')
    .with_columns(pl.col('n_stops').fill_null(0))
    .sort('n_stops', descending=True)
    .unique(subset=['route_short_name', 'direction_id'], keep='first')
    .sort(['route_short_name', 'direction_id'])
)

# ── Route geometry (~300 points per shape) ───────────────────────────────────
shape_ids = rep_trips['shape_id'].drop_nulls().to_list()
shapes_pd = (
    pl.scan_parquet(RAW_GTFS / 'gtfs_tram_shapes.parquet')
    .filter((pl.col('year') == '2025') & pl.col('shape_id').is_in(shape_ids))
    .select(['shape_id', 'shape_pt_lat', 'shape_pt_lon', 'shape_pt_sequence'])
    .sort(['shape_id', 'shape_pt_sequence'])
    .collect()
    .to_pandas()
)
shapes_dict = {}
for sid, grp in shapes_pd.groupby('shape_id', sort=True):
    shapes_dict[sid] = list(zip(grp['shape_pt_lat'], grp['shape_pt_lon']))

# ── Sorted line list ──────────────────────────────────────────────────────────
tram_lines = sorted(
    rep_trips['route_short_name'].unique().to_list(),
    key=lambda x: int(x) if x.isdigit() else 999
)

print(f'✓ Data loaded')
print(f'  Tram lines:   {len(tram_lines)}')
print(f'  Rep. trips:   {len(rep_trips)} (1 per line × direction)')
print(f'  Stop-times:   {len(stop_times_df):,} rows')
print(f'  Shapes:       {len(shapes_dict)} geometries')

## Overview

In [ ]:
fig = go.Figure()
for ln in reversed(tram_lines):
    color = LINE_COLORS.get(ln, '#999999')
    text_color = LINE_TEXT_COLORS.get(ln, '#FFFFFF')
    sub = rep_trips.filter(pl.col('route_short_name') == ln)
    dirs = {row['direction_id']: row['trip_headsign'] for row in sub.to_dicts()}
    dest_0 = (dirs.get(0) or '—').replace('Zürich, ', '')
    dest_1 = (dirs.get(1) or '—').replace('Zürich, ', '')
    label = f'{dest_1} ↔ {dest_0}'
    fig.add_trace(go.Bar(
        y=[f'L{ln}'], x=[1],
        orientation='h',
        marker_color=color,
        text=[label],
        textposition='inside',
        insidetextanchor='middle',
        hovertemplate=f'<b>Line {ln}</b><br>{label}<extra></extra>',
        showlegend=False,
        textfont=dict(color=text_color, size=11),
    ))
fig.update_layout(
    title='VBZ Tram Lines 2025 — Official Colors & Endpoints',
    xaxis=dict(showticklabels=False, showgrid=False, zeroline=False, range=[0, 1.05]),
    yaxis=dict(showgrid=False),
    height=600,
    barmode='stack',
    margin=dict(l=60, r=20, t=60, b=30),
    plot_bgcolor='white',
    paper_bgcolor='white',
)
fig.show()

## Line Directory

In [ ]:
# Stop counts per direction — flags asymmetric lines
rows = []
for ln in tram_lines:
    sub = rep_trips.filter(pl.col('route_short_name') == ln)
    dirs = {row['direction_id']: row for row in sub.to_dicts()}
    n0 = dirs.get(0, {}).get('n_stops', 0) or 0
    n1 = dirs.get(1, {}).get('n_stops', 0) or 0
    h0 = (dirs.get(0, {}).get('trip_headsign') or '—').replace('Zürich, ', '')
    h1 = (dirs.get(1, {}).get('trip_headsign') or '—').replace('Zürich, ', '')
    diff = abs(n0 - n1)
    flag = ' ⚠' if diff >= 4 else ''
    rows.append({'Line': f'L{ln}', 'dir=0 stops': n0, 'dir=1 stops': n1, 'Δ': diff,
                 '→ dir=0': h0, '→ dir=1': h1, '': flag})

import pandas as pd
df = pd.DataFrame(rows)
print('VBZ Tram Lines 2025 — Stop Counts per Direction')
print('(⚠ = asymmetric route, see Architecture note)')
print()
print(df.to_string(index=False))

## Helper Functions

In [ ]:
def get_line_stops(line_name: str, direction: int = 0) -> list[dict]:
    sub = rep_trips.filter(
        (pl.col('route_short_name') == line_name) &
        (pl.col('direction_id') == direction)
    )
    if sub.height == 0:
        return []
    trip_id = sub.row(0, named=True)['trip_id']
    trip_st = stop_times_df.filter(pl.col('trip_id') == trip_id).sort('stop_sequence')
    result = []
    for row in trip_st.to_dicts():
        info = stop_lookup.get(row['stop_id'])
        if info:
            result.append({
                'seq': int(row['stop_sequence']),
                'name': str(info['stop_name']).replace('Zürich, ', ''),
                'lat': float(info['stop_lat']),
                'lon': float(info['stop_lon']),
            })
    return result


def plot_line_map(line_name: str) -> go.Figure:
    color = LINE_COLORS.get(line_name, '#999999')
    fig = go.Figure()
    for direction in [0, 1]:
        sub = rep_trips.filter(
            (pl.col('route_short_name') == line_name) &
            (pl.col('direction_id') == direction)
        )
        if sub.height == 0:
            continue
        row = sub.row(0, named=True)
        headsign = (row.get('trip_headsign') or f'Direction {direction}').replace('Zürich, ', '')
        stops = get_line_stops(line_name, direction)
        if not stops:
            continue
        is_main = (direction == 0)
        opacity = 1.0 if is_main else 0.6
        shape_id = row.get('shape_id')
        if shape_id and shape_id in shapes_dict:
            pts = shapes_dict[shape_id]
            fig.add_trace(go.Scattermap(
                lat=[p[0] for p in pts],
                lon=[p[1] for p in pts],
                mode='lines',
                line=dict(width=4 if is_main else 2, color=color),
                opacity=opacity,
                hoverinfo='skip',
                showlegend=False,
            ))
        texts = [f'<b>{s["name"]}</b><br>Stop {s["seq"]}' for s in stops]
        fig.add_trace(go.Scattermap(
            lat=[s['lat'] for s in stops],
            lon=[s['lon'] for s in stops],
            mode='markers',
            marker=dict(size=9 if is_main else 6, color=color, opacity=opacity),
            text=texts,
            hovertemplate='%{text}<extra></extra>',
            name=f'→ {headsign}',
        ))
    main_stops = get_line_stops(line_name, 0) or get_line_stops(line_name, 1)
    if main_stops:
        mid = main_stops[len(main_stops) // 2]
        center = dict(lat=mid['lat'], lon=mid['lon'])
    else:
        center = dict(lat=47.378, lon=8.540)
    fig.update_layout(
        map=dict(style='carto-positron', center=center, zoom=12),
        margin=dict(l=0, r=0, t=40, b=0),
        height=500,
        title=f'Line {line_name} — Route 2025',
        showlegend=True,
        legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.85)', borderwidth=1),
    )
    return fig


print('✓ get_line_stops() and plot_line_map() ready')

## Export

In [ ]:
# Export ordered stop sequences for all lines and directions.
# Output: data/processed/tramlines_stops.parquet
# Schema: line_name, direction_id, headsign, stop_sequence, stop_name, stop_lat, stop_lon
# Use case: dashboard access without re-running GTFS join logic.

export_rows = []
for ln in tram_lines:
    sub = rep_trips.filter(pl.col('route_short_name') == ln)
    for trip_row in sub.to_dicts():
        direction = trip_row['direction_id']
        headsign = (trip_row.get('trip_headsign') or f'Direction {direction}').replace('Zürich, ', '')
        stops = get_line_stops(ln, direction)
        for s in stops:
            export_rows.append({
                'line_name': ln,
                'direction_id': int(direction),
                'headsign': headsign,
                'stop_sequence': s['seq'],
                'stop_name': s['name'],
                'stop_lat': s['lat'],
                'stop_lon': s['lon'],
            })

tramlines_stops = pl.DataFrame(export_rows)
out_path = PATHS['processed'] / 'tramlines_stops.parquet'
tramlines_stops.write_parquet(out_path)
print(f'✓ Exported → data/processed/tramlines_stops.parquet')
print(f'  Rows:   {len(tramlines_stops):,}')
print(f'  Lines:  {tramlines_stops["line_name"].n_unique()}')
print()
print(tramlines_stops.group_by(['line_name', 'direction_id']).agg(pl.len().alias('n_stops')).sort(['line_name', 'direction_id']))

## Line Details

In [ ]:
for ln in tram_lines:
    color = LINE_COLORS.get(ln, '#999999')
    sub = rep_trips.filter(pl.col('route_short_name') == ln)
    dirs = {row['direction_id']: row for row in sub.to_dicts()}
    dest_0 = (dirs.get(0, {}).get('trip_headsign') or '—').replace('Zürich, ', '')
    dest_1 = (dirs.get(1, {}).get('trip_headsign') or '—').replace('Zürich, ', '')
    n0 = dirs.get(0, {}).get('n_stops') or 0
    n1 = dirs.get(1, {}).get('n_stops') or 0
    # Line header
    style = f'color:{color}; border-left:6px solid {color}; padding-left:12px; margin-top:40px; font-family:sans-serif'
    display(HTML(f'<h2 style="{style}">Line {ln} &nbsp;—&nbsp; {dest_1} ↔ {dest_0}</h2>'))
    # Asymmetry note
    if n0 != n1:
        diff = abs(n0 - n1)
        base = (
            f'Richtung 0: {n0} Halte, Richtung 1: {n1} Halte, Δ={diff} — '
            f'direction_id=1 (→ {dest_1}) deckt eine längere Strecke ab als direction_id=0 (→ {dest_0}). '
        )
        if diff <= 3:
            detail = (
                'Normale Asymmetrie: Trams nehmen auf Einbahnstraßen-Abschnitten in der Innenstadt '
                'unterschiedliche Wege, oder eine Richtung fährt eine Schleife an der Endstation (+1 Halt).'
            )
        elif ln == '8':
            detail = (
                'Die GTFS-Daten 2025 bilden die offizielle Strecke vor dem 14. Dezember 2025 ab '
                '(Klusplatz ↔ Wollishoferplatz mit Wollishofen-Extension in dir=1). '
                'Am 14.12.2025 trat der grösste Fahrplanwechsel in der VBZ-Geschichte in Kraft: '
                'L8 fährt seitdem Hardturm ↔ Kirche Fluntern (neues Tramnetz Süd). '
                'Methodisch konsistent: der Verspätungs-Hauptdatensatz (2023–2025) enthält '
                'ebenfalls Daten unter dieser alten Streckenführung.'
            )
        else:
            detail = (
                'Strukturelle Asymmetrie: ein Streckenabschnitt ist im GTFS 2025 nur in einer '
                'Fahrtrichtung vollständig hinterlegt. Kein Datenfehler — der Algorithmus ignoriert '
                'service_id/Kalender und wählt immer den längsten Trip der jeweiligen Richtung.'
            )
        display(HTML(
            f'<p style="font-family:sans-serif; color:#777; margin:4px 0 12px 20px; '
            f'font-size:0.88em; font-style:italic">ℹ {base}{detail}</p>'
        ))
    # Stop sequences for both directions
    for direction, dest in [(0, dest_0), (1, dest_1)]:
        stops = get_line_stops(ln, direction)
        if stops:
            print(f'  → {dest}  ({len(stops)} Halte)')
            for s in stops:
                print(f'    {s["seq"]:2d}. {s["name"]}')
        else:
            print(f'  Richtung {direction}: keine Daten')
        print()
    fig = plot_line_map(ln)
    fig.show()
    print()
    print('─' * 60)
    print()